# Case Study §9 — Merge LoRA → GGUF (Part B: deploy at the edge)

Runnable twin of [`09_merge_and_gguf.py`](09_merge_and_gguf.py). On-device runtimes consume GGUF, and
`convert_hf_to_gguf.py` does **not** accept PEFT adapters — so the order is: **merge LoRA → F16 GGUF →
quantize (Q4_K_M)**.

- **Merge** always runs (PEFT `merge_and_unload` → a standalone safetensors model). We merge the §5 SFT
  adapter (it doesn't touch the tied embeddings, so the merge is clean).
- **GGUF** needs llama.cpp. If it's not installed we print the exact recipe and emit a Modelfile so
  Ollama (§10) can import the merged safetensors directly.

In [ ]:
MODE = "trial"
import importlib.util, pathlib, sys
HERE = pathlib.Path.cwd()
root = HERE if (HERE / "09_merge_and_gguf.py").exists() else HERE / "case_study"
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s9", root / "09_merge_and_gguf.py")
s9 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s9)
info = s9.run()

In [ ]:
import pathlib
merged = pathlib.Path(info['merged'])
assert (merged / 'config.json').exists() and (merged / 'model.safetensors').exists(), 'merged model present'
assert (merged / 'Modelfile').exists(), 'Modelfile emitted for Ollama'
print('\u2713 §9 verified: standalone merged model + Modelfile ready.')
print('GGUF:', info['gguf'] or 'skipped (no llama.cpp) \u2014 Ollama can import the merged model directly.')
print('Next: \u00a710 deploy with Ollama.')